In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_PATH = "/content/drive/My Drive/ai_programming/final/model/"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

model.eval()

print("Model loaded!")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded!


In [ ]:
# def predict_single(text):
#     inputs = tokenizer(
#         text,
#         return_tensors="pt",
#         truncation=True,
#         padding=True,
#         max_length=128
#     )

#     with torch.no_grad():
#         outputs = model(**inputs)
#         probs = torch.softmax(outputs.logits, dim=1)

#     ai_score = probs[0][1].item()
#     return ai_score

def predict_single(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)

    ai_score = probs[0][1].item()

    # compress overconfidence
    ai_score = (ai_score - 0.5) * 0.5 + 0.5

    # Penalize short text (likely human tweets)
    if len(text.split()) < 10:
        ai_score *= 0.7

    return ai_score

In [ ]:
def classify_account(posts, threshold=0.7):
    scores = [predict_single(p) for p in posts if p.strip() != ""]

    if len(scores) == 0:
        return {
            "label": "No valid posts",
            "average_score": 0,
            "scores": []
        }

    avg_score = sum(scores) / len(scores)

    if avg_score > threshold:
        label = "Likely AI Bot"
    else:
        label = "Likely Human"

    return {
        "label": label,
        "average_score": avg_score,
        "scores": scores
    }

In [ ]:

posts = [
    "As the world begins to grow ANX expand, many people have found that living without cars can be more beneficial. Cities ANX suburbs around the world, have found that reducing car usage has a great number of advantages. This is reducing pollution, ANX is reducing stress. The world is shifting into a new way of life, not involving cars. Cars release a large about of greenhouse gases into the atmosphere every single Day, ANX this is harming the environment. Fxperts explain that, efforts to Drastically reduce greenhouse gas emissions from tailpipes (Rosenthal), have been taking place in suburban communities in Germany. This has resulted with a, 12% of greenhouse gas emissions in Europe... ANX up to 50% in car intensive areas in the United States (Rosenthal). Also, Paris has seen rerecord amounts of pollution ANX smog in the city, so the global city has limited motorists' Driving on two Days of the week, Monday ANX Tuesday. The climate in Paris, [causes] the warm layer of air to trap car emissions (Duffer), throughout the atmosphere in the city, but the limitation on motorists at the beginning of the week has allowed the smog to clear up. Helping the environment is not the only benefit of reducing car usage, yet it can also relieve stress in your everyday life. There is no need to stress out over traffic, finding somewhere to park, or even paying the expenses that come with owning a car. In Germany, most residents On't own cars, ANX Earn Walter explained that she is, much happier this way (Rosenthal), because she Doesn have to worry about traffic. She has enjoyed a healthier lifestyle that is stress-free, by walking or biking to the places she needs to be. In Colombia, there is a Day that citizens will not use cars, once a year. Businessman Carlos Arturo Plaza agrees that, \"it's a good opportunity to take away stress\" (Silky). There is no need to worry about rush hour, or other traffic. Others On\'t see the advantages of reducing car usage. Delivery companies complain about the loss in revenue, yet they aren\'t thinking about the benefits. They can save their employees from stressing out, or even help the environment. Reducing car usage would be a huge cultural shift, that has many advantages."
      ]

print(classify_account(posts))

{'label': 'Likely Human', 'average_score': 0.2502714776492212, 'scores': [0.2502714776492212]}


In [ ]:
# Or save this whole code block as app.py and run to run streamlit on ur local
# > pip install streamlit transformers torch
# > streamlit run app.py

import streamlit as st
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_PATH = "model"

@st.cache_resource
def load_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
    model.eval()
    return tokenizer, model

tokenizer, model = load_model()


def predict_single(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)

    score = probs[0][1].item()

    # same calibration
    score = (score - 0.5) * 0.5 + 0.5
    if len(text.split()) < 10:
        score *= 0.7

    return score


def classify_account(posts):
    scores = [predict_single(p) for p in posts if p.strip() != ""]
    avg = sum(scores) / len(scores)

    label = "Likely AI Bot" if avg >= 0.74 else "Likely Human"
    return avg, label, scores


# UI
st.title("🤖 AI Bot Detector")

st.write("Paste multiple posts (one per line):")

user_input = st.text_area("Posts")

if st.button("Analyze"):
    posts = user_input.split("\n")

    avg, label, scores = classify_account(posts)

    st.subheader(label)
    st.write(f"Confidence: {avg:.2f}")

    st.subheader("Post-level scores:")
    for i, s in enumerate(scores):
        st.write(f"Post {i+1}: {s:.2f}")